### Imports

In [77]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import xgboost as xgb

### Load Data

In [78]:
df = pd.read_csv("/Users/jonathankipping/code/Petrol/price_history.csv")


### Inspection:

In [79]:
print(df.head())
print(df.info())
print(df.describe())

       id                                            node_id    fuel_type  \
0  167270  a764a8afa258239622aa57e361b18547af89dace23e837...          E10   
1  167271  a764a8afa258239622aa57e361b18547af89dace23e837...  B7_STANDARD   
2  167272  ab101b267dc0ea3e9e314942bc86ab7fd8a97652d583cd...          E10   
3  167273  8d057bb47fb524f2bad8c08cb8bf0c2dc4a03ae46bc9e4...          E10   
4  167274  8d057bb47fb524f2bad8c08cb8bf0c2dc4a03ae46bc9e4...  B7_STANDARD   

   price_pence               recorded_at         source_updated_at  
0        147.9  2026-03-15T07:08:01.297Z  2026-03-15T07:04:03.000Z  
1        165.9  2026-03-15T07:08:01.297Z  2026-03-15T07:04:03.000Z  
2        143.9  2026-03-15T07:08:01.297Z  2026-03-15T07:04:03.000Z  
3        144.9  2026-03-15T07:08:01.297Z  2026-03-15T07:04:03.000Z  
4        164.9  2026-03-15T07:08:01.297Z  2026-03-15T07:04:03.000Z  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 371010 entries, 0 to 371009
Data columns (total 6 columns):
 #   Column  

### Drop Irrelevant Columns

In [80]:
df = df.drop(columns=['id', 'node_id', 'source_updated_at'])

### Define Features and target:

In [81]:
X = df.drop(columns=['price_pence'])
y = df['price_pence']

### Convert recorded_at to datetime and extract numeric features

In [82]:
X['recorded_at'] = pd.to_datetime(X['recorded_at'])
X['year'] = X['recorded_at'].dt.year
X['month'] = X['recorded_at'].dt.month
X['day'] = X['recorded_at'].dt.day
X['weekday'] = X['recorded_at'].dt.weekday
X = X.drop(columns=['recorded_at'])

### Encode Categorical features


In [83]:
X = pd.get_dummies(X, columns=['fuel_type'], drop_first=True)


### Scale numeric features

In [84]:
num_cols = X.select_dtypes(include='number').columns
scaler = StandardScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])

### Test/Train Split

In [85]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

### Initialise XGBoost regressor  

In [86]:
model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    tree_method='hist'  # fast for large datasets
)

### Train Model

In [87]:
model.fit(X_train, y_train)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


### Function to predict price on a given date

In [ ]:
# -------------------------
# Predict price for a given date
# -------------------------

def predict_price(model, scaler, date_str, fuel_type='E10'):
    """
    Predict the petrol price (in pence) for a given date and fuel type using a trained model.

    Parameters
    ----------
    model : object
        The trained regression model (e.g., XGBoost or RandomForest) used for prediction.
    scaler : sklearn.preprocessing.StandardScaler
        The fitted scaler used to scale numeric features during training.
    date_str : str
        The target date for prediction, in 'YYYY-MM-DD' format.
    fuel_type : str, optional (default='E10')
        The fuel type to predict (must match one-hot encoded columns in training data,
        e.g., 'E10', 'B7_STANDARD', etc.).

    Returns
    -------
    float
        The predicted petrol price (in pence) for the specified date and fuel type.

    Notes
    -----
    - The function automatically extracts numeric features from the date:
        year, month, day, and weekday.
    - Fuel type is encoded using the same one-hot columns as the training data.
    - Only numeric features that were scaled during training are transformed with the scaler.
    - The function ensures the feature order matches the training set before prediction.
    """

    date = pd.to_datetime(date_str)

    # Create future dataframe with numeric features
    future_df = pd.DataFrame({
        'year': [date.year],
        'month': [date.month],
        'day': [date.day],
        'weekday': [date.weekday()]
    })

    # Add one-hot encoded fuel columns
    for col in [c for c in X.columns if c.startswith('fuel_type_')]:
        future_df[col] = 1 if col == f'fuel_type_{fuel_type}' else 0

    # Add any missing columns from training set
    for col in X.columns:
        if col not in future_df.columns:
            future_df[col] = 0
    future_df = future_df[X.columns]  # reorder columns

    # Only scale the numeric columns used in training
    numeric_cols = [c for c in num_cols if c in future_df.columns]
    future_df[numeric_cols] = scaler.transform(future_df[numeric_cols])

    # Predict
    predicted_price = model.predict(future_df)[0]
    return predicted_price

# Example usage
date_input = "2026-12-31"
fuel_input = "E10"
predicted = predict_price(model, scaler, date_input, fuel_input)
print(f"Predicted price for {fuel_input} on {date_input}: {predicted:.2f} pence")

Predicted price for E10 on 2026-12-31: 159.34 pence


### Evaluate on Test Set

In [88]:
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))  # manual RMSE

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")

MAE: 6.09
RMSE: 30.08


### Predict Future Price

In [89]:
future_date = pd.to_datetime("2026-04-15")
future_df = pd.DataFrame({
    'year': [future_date.year],
    'month': [future_date.month],
    'day': [future_date.day],
    'weekday': [future_date.weekday()],
    'fuel_type_B7_STANDARD': [0],  # One-hot: adjust for your target fuel type
    'fuel_type_E10': [1]           # Example: predicting E10
})

# Ensure all columns match the trained data
for col in X.columns:
    if col not in future_df.columns:
        future_df[col] = 0
future_df = future_df[X.columns]  # Reorder columns

# Scale numeric features
future_df[num_cols] = scaler.transform(future_df[num_cols])

# Predict
future_price = model.predict(future_df)
print(f"Predicted price (pence) for 2026-04-15: {future_price[0]:.2f}")

Predicted price (pence) for 2026-04-15: 157.46
